In [3]:
import torch
from torchvision import transforms, models  # Import models here
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader
from mtcnn import MTCNN
import os
import torch.optim as optim

# Initialize MTCNN detector
detector = MTCNN()

# Function to extract faces from video frames (already provided)
def extract_faces_from_video(video_path):
    frames = load_video_frames(video_path)  # This should be a list of PIL images (frames)
    faces = []
    for frame in frames:
        detected_faces = detector.detect_faces(np.array(frame))  # Detect faces using MTCNN
        for face in detected_faces:
            x, y, w, h = face['box']
            face_region = frame.crop((x, y, x + w, y + h))  # Extract face region
            faces.append(face_region)
    return faces

def load_video_frames(video_path):
    # Simple placeholder function to simulate video frame extraction
    frames = []
    for filename in os.listdir(video_path):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            img_path = os.path.join(video_path, filename)
            img = Image.open(img_path)
            frames.append(img)
    return frames

# Dataset class (same as you provided)
class FaceSwapDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx])
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Example image paths (replace with your actual paths)
real_images = ["C:\\Users\\Brund\\Downloads\\train_images\\real_and_fake_face\\training_real\\real_01080.jpg",
               "C:\\Users\\Brund\\Downloads\\train_images\\real_and_fake_face\\training_real\\real_01081.jpg"]
fake_images = ["C:\\Users\\Brund\\Downloads\\train_images\\real_and_fake_face\\training_fake\\mid_9_1110.jpg",
               "C:\\Users\\Brund\\Downloads\\train_images\\real_and_fake_face\\training_fake\\mid_7_1100.jpg"]

# Assign labels (0 for real, 1 for fake)
image_paths = real_images + fake_images
labels = [0] * len(real_images) + [1] * len(fake_images)

# Define transformations for preprocessing (resizing, normalization)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create dataset and dataloaders
dataset = FaceSwapDataset(image_paths=image_paths, labels=labels, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Load the pre-trained ResNet50 model
model = models.resnet50(pretrained=True)  # This is where the error occurred
model.fc = torch.nn.Linear(model.fc.in_features, 1)  # Binary classification: real (0) vs fake (1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define loss function and optimizer
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device).float()

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images).squeeze()  # Squeeze to remove extra dimensions
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        preds = torch.round(torch.sigmoid(outputs))  # Round predictions to get 0 or 1
        correct_preds += (preds == labels).sum().item()
        total_preds += labels.size(0)

    avg_loss = running_loss / len(dataloader)
    accuracy = correct_preds / total_preds

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")


# Prediction function
def predict_image(image_path):
    model.eval()  # Set model to evaluation mode
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Load and preprocess the image
    img = Image.open(image_path)
    img = transform(img).unsqueeze(0).to(device)  # Add batch dimension and move to device

    # Predict
    with torch.no_grad():
        output = model(img)
        prediction = torch.sigmoid(output).item()  # Sigmoid to get probability

    # Print the prediction
    if prediction < 0.5:
        print(f"The image is REAL with confidence {1 - prediction:.4f}")
    else:
        print(f"The image is FAKE with confidence {prediction:.4f}")

# Example: Test the prediction function with a real or fake image
image_path = "C:\\Users\\Brund\\Downloads\\train_images\\real_and_fake_face\\training_real\\real_01080.jpg" 
predict_image(image_path)


Epoch [1/5], Loss: 0.7171, Accuracy: 0.5000
Epoch [2/5], Loss: 0.2497, Accuracy: 1.0000
Epoch [3/5], Loss: 0.1152, Accuracy: 1.0000
Epoch [4/5], Loss: 0.0613, Accuracy: 1.0000
Epoch [5/5], Loss: 0.0353, Accuracy: 1.0000
The image is REAL with confidence 0.9501
